# **SQL in R**

## **1. Install and load packages**

In [ ]:
# Install packages
install.packages("sqldf")
install.packages("ggplot2")
install.packages("dplyr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
# Load libraries
library(sqldf)
library(ggplot2)
library(dplyr)

print("Packages loaded")

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "Packages loaded"


## **2. Load the cleaned CSV files**

In [16]:
# Load each cleaned CSV
deliveries <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/deliveries_clean.csv", stringsAsFactors = FALSE)
orders     <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/orders_clean.csv",     stringsAsFactors = FALSE)
drivers    <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/drivers_clean.csv",    stringsAsFactors = FALSE)
complaints <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/complaints_clean.csv", stringsAsFactors = FALSE)
hubs       <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/hubs_clean.csv",       stringsAsFactors = FALSE)
vehicles   <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/vehicles_clean.csv",   stringsAsFactors = FALSE)
incidents  <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/incidents_clean.csv",  stringsAsFactors = FALSE)
customers  <- read.csv("https://raw.githubusercontent.com/Habishalini/NorthStar-Database_Analytics-Coursework/refs/heads/main/Cleaned%20Datasets/customers_clean.csv",  stringsAsFactors = FALSE)

# Confirm row counts
cat("deliveries:", nrow(deliveries), "rows\n")
cat("orders:    ", nrow(orders),     "rows\n")
cat("drivers:   ", nrow(drivers),    "rows\n")
cat("complaints:", nrow(complaints), "rows\n")
cat("hubs:      ", nrow(hubs),       "rows\n")
cat("vehicles:  ", nrow(vehicles),   "rows\n")
cat("incidents: ", nrow(incidents),  "rows\n")
cat("customers: ", nrow(customers),  "rows\n")

deliveries: 950 rows
orders:     1250 rows
drivers:    170 rows
complaints: 320 rows
hubs:       8 rows
vehicles:   120 rows
incidents:  280 rows
customers:  650 rows


## **3. Query 1 - Total and failed deliveries per zone**
Find which pickup zone has the most failed deliveries

In [17]:
query1 <- sqldf("
SELECT pickup_zone, COUNT(*) AS total_deliveries,
  SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
GROUP BY pickup_zone
ORDER BY failed_deliveries DESC
")

print(query1)

  pickup_zone total_deliveries failed_deliveries
1     Central              174                33
2       North              135                22
3        East              156                19
4   Riverside              119                18
5        West              114                14
6       South              139                14
7     Airport              113                12


## **4. Query 2 - Average rating per zone**
Find which zone has the lowest customer ratings

In [18]:
query2 <- sqldf("
SELECT
  pickup_zone,
  COUNT(*) AS total_deliveries,
  ROUND(AVG(customer_rating_post_delivery), 2) AS average_rating
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
GROUP BY pickup_zone
ORDER BY average_rating ASC
")

print(query2)

  pickup_zone total_deliveries average_rating
1     Central              174           3.56
2   Riverside              119           3.86
3       North              135           3.90
4        West              114           3.90
5        East              156           3.91
6     Airport              113           3.98
7       South              139           4.05


## **5. Query 3 - Complaints by type and severity**
Find which complaint types take longest to resolve

In [19]:
query3 <- sqldf("
SELECT
  complaint_type,
  severity,
  COUNT(*) AS number_of_complaints,
  ROUND(AVG(resolution_days), 2) AS average_days_to_resolve
FROM complaints
GROUP BY complaint_type, severity
ORDER BY complaint_type, severity
")

print(query3)

      complaint_type severity number_of_complaints average_days_to_resolve
1           AppIssue     High                   13                   13.92
2           AppIssue      Low                   15                    6.07
3           AppIssue   Medium                   25                    7.36
4            Billing     High                    4                   12.00
5            Billing      Low                    3                    8.00
6            Billing   Medium                    9                    5.78
7             Damage     High                    7                   15.43
8             Damage      Low                    6                    6.83
9             Damage   Medium                    2                   10.50
10             Delay     High                   18                   12.44
11             Delay      Low                   27                    6.48
12             Delay   Medium                   56                    5.96
13   DriverBehaviour     

## **6. Query 4 - Top 10 worst drivers**
Find which 10 drivers have the most failed deliveries

In [20]:
query4 <- sqldf("
SELECT
  drivers.driver_id, drivers.driver_rating, drivers.years_experience,
  COUNT(*) AS total_deliveries,
  SUM(CASE WHEN deliveries.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
  ROUND(AVG(deliveries.manual_route_override_count), 2) AS average_overrides
FROM drivers
JOIN deliveries ON drivers.driver_id = deliveries.driver_id
GROUP BY drivers.driver_id, drivers.driver_rating, drivers.years_experience
ORDER BY failed_deliveries DESC LIMIT 10
")

print(query4)

   driver_id driver_rating years_experience total_deliveries failed_deliveries
1       D024          3.35                8                8                 4
2       D104          3.45               15                7                 4
3       D133          3.99               12               12                 4
4       D004          4.75               13                9                 3
5       D010          3.95                8                7                 3
6       D055          5.00               15               10                 3
7       D083          4.16               12                9                 3
8       D092          4.24               15                5                 3
9       D108          4.33               10               11                 3
10      D131          4.26                9                9                 3
   average_overrides
1               1.13
2               1.71
3               0.92
4               0.78
5               0.86
6    

## **7. Query 5 - Hub performance**
Find which hub dispatches the most failed deliveries

In [21]:
query5 <- sqldf("
SELECT
  hubs.hub_id, hub_name, zone,
  COUNT(*) AS total_deliveries,
  SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
  ROUND(AVG(fuel_or_charge_cost), 2) AS average_cost
FROM hubs
JOIN deliveries ON hubs.hub_id = deliveries.hub_id
GROUP BY hubs.hub_id, hub_name, zone
ORDER BY failed_deliveries DESC
")

print(query5)

  hub_id       hub_name      zone total_deliveries failed_deliveries
1    H08  Midtown Relay   Central              128                26
2    H05   Central Core   Central              115                23
3    H01 North Exchange     North              136                17
4    H04      West Gate      West              127                16
5    H06    Airport Hub   Airport              104                15
6    H07  Riverside Hub Riverside              115                14
7    H03      East Dock      East              119                11
8    H02     South Link     South              106                10
  average_cost
1        11.71
2        13.69
3        12.76
4        13.17
5        13.32
6        12.92
7        12.74
8        12.57


## **8. Query 6 - Combined zone summary**
One table showing failure rate, rating, and profit per zone

In [22]:
query6 <- sqldf("
SELECT
  pickup_zone,
  COUNT(*) AS total_deliveries,
  SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
  ROUND(AVG(customer_rating_post_delivery), 2) AS average_rating,
  ROUND(AVG(order_value - fuel_or_charge_cost), 2) AS average_profit
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
GROUP BY pickup_zone HAVING COUNT(*) > 50 ORDER BY failed_deliveries DESC
")

print(query6)

  pickup_zone total_deliveries failed_deliveries average_rating average_profit
1     Central              174                33           3.56          75.70
2       North              135                22           3.90          78.15
3        East              156                19           3.91          80.88
4   Riverside              119                18           3.86          77.88
5        West              114                14           3.90          77.11
6       South              139                14           4.05          79.89
7     Airport              113                12           3.98          84.64


## **9. Set up the database for indexing**

In [23]:
# Install DBI and RSQLite for indexing
install.packages("DBI", quiet = TRUE)
install.packages("RSQLite", quiet = TRUE)

library(DBI)
library(RSQLite)

# Open an empty database in memory
database <- dbConnect(SQLite(), ":memory:")

# Copy ALL the data frames into the database
dbWriteTable(database, "deliveries", deliveries)
dbWriteTable(database, "orders", orders)
dbWriteTable(database, "complaints", complaints)
dbWriteTable(database, "drivers", drivers)
dbWriteTable(database, "hubs", hubs)
dbWriteTable(database, "vehicles", vehicles)
dbWriteTable(database, "incidents", incidents)
dbWriteTable(database, "customers", customers)

print(dbListTables(database))

[1] "complaints" "customers"  "deliveries" "drivers"    "hubs"      
[6] "incidents"  "orders"     "vehicles"  


## **10. Check query speed BEFORE indexing**

In [24]:
# Check database plans to find data without an index

before_indexing <- dbGetQuery(database, "
EXPLAIN QUERY PLAN
SELECT delivery_id, delivery_status, pickup_zone
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
WHERE pickup_zone = 'Central'
")

print("BEFORE indexing:")
print(before_indexing)

[1] "BEFORE indexing:"
  id parent notused
1  3      0     216
2 17      0      53
                                                         detail
1                                                   SCAN orders
2 SEARCH deliveries USING AUTOMATIC COVERING INDEX (order_id=?)


## **11. Create indexes**

In [25]:
# Create an index on pickup_zone (filter by)
dbExecute(database, "CREATE INDEX my_zone_index ON orders(pickup_zone)")

# Create an index on order_id (join on)
dbExecute(database, "CREATE INDEX my_order_index ON deliveries(order_id)")

print("Indexes created")

[1] 0

[1] 0

[1] "Indexes created"


## **12. Check query speed AFTER indexing**

In [26]:
# Run the same query plan check - should be faster now

after_indexing <- dbGetQuery(database, "
EXPLAIN QUERY PLAN
SELECT delivery_id, delivery_status, pickup_zone
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
WHERE pickup_zone = 'Central'
")

print("AFTER indexing:")
print(after_indexing)

[1] "AFTER indexing:"
  id parent notused                                                    detail
1  5      0      61   SEARCH orders USING INDEX my_zone_index (pickup_zone=?)
2 10      0      61 SEARCH deliveries USING INDEX my_order_index (order_id=?)


## **13. The "system fragmentation" query**

In [27]:
# First add the complaints dataframe into the SQLite database
# This creates a SQL table called "complaints"

dbWriteTable(
  database,
  "complaints",
  complaints,
  overwrite = TRUE
)

# Find deliveries marked OnTime but that have customer complaints

fragmentation <- dbGetQuery(database, "
SELECT
  deliveries.delivery_id,
  delivery_status,
  pickup_zone
FROM deliveries
JOIN orders ON deliveries.order_id = orders.order_id
WHERE delivery_status = 'OnTime'
  AND orders.order_id IN (SELECT order_id FROM complaints)
")

print("Number of OnTime deliveries that also have complaints:")
print(nrow(fragmentation))

print("First 10 examples:")
print(head(fragmentation, 10))

[1] "Number of OnTime deliveries that also have complaints:"
[1] 131
[1] "First 10 examples:"
   delivery_id delivery_status pickup_zone
1      DL00671          OnTime   Riverside
2      DL00201          OnTime   Riverside
3      DL00606          OnTime        East
4      DL00236          OnTime     Airport
5      DL00110          OnTime       South
6      DL00225          OnTime       South
7      DL00237          OnTime     Central
8      DL00111          OnTime       North
9      DL00756          OnTime     Central
10     DL00578          OnTime        East


# **14. Close connection**

In [28]:
dbDisconnect(database)
print("Database closed")

[1] "Database closed"
